In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from mlflow.tracking import MlflowClient
from datetime import datetime

print("=" * 80)
print("🔍 DATA AND TABLE VERIFICATION")
print("=" * 80)
print(f"Verification Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

## Step 1: Verify Input Data
Check if the training data CSV file exists and is readable

In [ ]:
# Configuration - Update these if your paths are different
INPUT_DATA_PATH = "/Volumes/pru/pac_mlops/test/nyctaxi-with-zipcodes.csv"
GROUND_TRUTH_TABLE = "pru.pac_mlops.ground_truth"
PREDICTIONS_TABLE = "pru.pac_mlops.predictions"
MODEL_NAME = "pru.pac_mlops.pac_mlops-model"
EXPECTED_ID_COL = "pickup_zip"
EXPECTED_GROUND_TRUTH_COL = "fare_amount"

print("\n📁 STEP 1: Verifying Input Data")
print("-" * 80)

try:
    # Try to read the CSV file
    input_df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(INPUT_DATA_PATH)
    )

    row_count = input_df.count()
    columns = input_df.columns

    print(f"✅ Input data file found: {INPUT_DATA_PATH}")
    print(f"   - Row count: {row_count:,}")
    print(f"   - Column count: {len(columns)}")
    print(
        f"   - Columns: {', '.join(columns[:10])}"
        + ("..." if len(columns) > 10 else "")
    )

    # Check for expected columns
    if EXPECTED_ID_COL in columns:
        print(f"   ✅ ID column '{EXPECTED_ID_COL}' found")
    else:
        print(f"   ❌ ID column '{EXPECTED_ID_COL}' NOT found")

    if EXPECTED_GROUND_TRUTH_COL in columns:
        print(f"   ✅ Ground truth column '{EXPECTED_GROUND_TRUTH_COL}' found")
    else:
        print(f"   ❌ Ground truth column '{EXPECTED_GROUND_TRUTH_COL}' NOT found")

    # Show sample data
    print("\n   📋 Sample data (first 5 rows):")
    input_df.limit(5).display()

except Exception as e:
    print(f"❌ ERROR reading input data: {str(e)}")
    print("   Please ensure the file exists and is accessible")

## Step 2: Verify Ground Truth Table
Check if the ground truth table exists and contains data

In [ ]:
print("\n📊 STEP 2: Verifying Ground Truth Table")
print("-" * 80)

try:
    # Check if table exists
    table_exists = spark.catalog.tableExists(GROUND_TRUTH_TABLE)

    if table_exists:
        print(f"✅ Ground truth table exists: {GROUND_TRUTH_TABLE}")

        # Get table info
        gt_df = spark.table(GROUND_TRUTH_TABLE)
        row_count = gt_df.count()
        columns = gt_df.columns

        print(f"   - Row count: {row_count:,}")
        print(f"   - Columns: {', '.join(columns)}")

        if row_count > 0:
            print(f"   ✅ Table is populated with {row_count:,} records")

            # Show sample data
            print("\n   📋 Sample data (latest 5 records):")
            gt_df.orderBy("timestamp", ascending=False).limit(5).display()

            # Show unique projects and batches
            print("\n   📈 Summary Statistics:")
            gt_df.groupBy("project_name").count().display()

        else:
            print(f"   ⚠️  Table exists but is EMPTY")
            print(f"   → The Feature Engineering job may not have run successfully")

    else:
        print(f"❌ Ground truth table does NOT exist: {GROUND_TRUTH_TABLE}")
        print(f"   → The Feature Engineering job has not created this table yet")

except Exception as e:
    print(f"❌ ERROR checking ground truth table: {str(e)}")

## Step 3: Verify Predictions Table
Check if the predictions table exists and contains data

In [ ]:
print("\n🎯 STEP 3: Verifying Predictions Table")
print("-" * 80)

try:
    # Check if table exists
    table_exists = spark.catalog.tableExists(PREDICTIONS_TABLE)

    if table_exists:
        print(f"✅ Predictions table exists: {PREDICTIONS_TABLE}")

        # Get table info
        pred_df = spark.table(PREDICTIONS_TABLE)
        row_count = pred_df.count()
        columns = pred_df.columns

        print(f"   - Row count: {row_count:,}")
        print(f"   - Columns: {', '.join(columns)}")

        if row_count > 0:
            print(f"   ✅ Table is populated with {row_count:,} records")

            # Show sample data
            print("\n   📋 Sample data (latest 5 records):")
            pred_df.orderBy("timestamp", ascending=False).limit(5).display()

            # Show summary by model version
            print("\n   📈 Summary by Model Version:")
            pred_df.groupBy("model_name", "model_version").count().display()

        else:
            print(f"   ⚠️  Table exists but is EMPTY")
            print(f"   → The Batch Inference job may not have run successfully")

    else:
        print(f"❌ Predictions table does NOT exist: {PREDICTIONS_TABLE}")
        print(f"   → The Batch Inference job has not created this table yet")

except Exception as e:
    print(f"❌ ERROR checking predictions table: {str(e)}")

## Step 4: Verify Model Registry
Check if the model exists with champion alias

In [ ]:
print("\n🏆 STEP 4: Verifying Model Registry")
print("-" * 80)

try:
    # Initialize MLflow client with Unity Catalog
    client = MlflowClient(registry_uri="databricks-uc")

    # Check if model exists
    try:
        # Try to get model by alias
        model_version = client.get_model_version_by_alias(MODEL_NAME, "champion")

        print(f"✅ Model exists: {MODEL_NAME}")
        print(f"   - Champion version: {model_version.version}")
        print(f"   - Status: {model_version.status}")
        print(f"   - Created: {model_version.creation_timestamp}")

        # Get all versions
        all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
        print(f"   - Total versions: {len(all_versions)}")

        # Show all aliases
        print(f"\n   📋 Model Aliases:")
        for version in all_versions[:5]:  # Show first 5
            aliases = version.aliases if hasattr(version, "aliases") else []
            print(
                f"      Version {version.version}: {', '.join(aliases) if aliases else 'No aliases'}"
            )

    except Exception as alias_error:
        print(f"❌ No model found with 'champion' alias")
        print(f"   Error: {str(alias_error)}")

        # Try to check if model exists at all
        try:
            all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
            if all_versions:
                print(f"\n   ℹ️  Model exists but has no 'champion' alias")
                print(f"   - Total versions: {len(all_versions)}")
                print(f"   → You need to assign 'champion' alias to a version")
            else:
                print(f"\n   ❌ Model does not exist: {MODEL_NAME}")
                print(f"   → The Training job has not created this model yet")
        except:
            print(f"\n   ❌ Model does not exist: {MODEL_NAME}")
            print(f"   → The Training job has not created this model yet")

except Exception as e:
    print(f"❌ ERROR checking model registry: {str(e)}")

## Summary and Recommendations

In [ ]:
print("\n" + "=" * 80)
print("📝 VERIFICATION SUMMARY")
print("=" * 80)

print(
    """
NEXT STEPS:

1. If input data verification FAILED:
   ✓ Ensure the CSV file exists at the specified path
   ✓ Check Databricks workspace permissions for the volume

2. If ground truth table is EMPTY or doesn't exist:
   ✓ Run the Feature Engineering workflow
   ✓ Check the CreateGroundTruth notebook logs for errors

3. If model doesn't exist or lacks 'champion' alias:
   ✓ Run the Training workflow to create the model
   ✓ Assign 'champion' alias to the best model version

4. If predictions table is EMPTY or doesn't exist:
   ✓ Ensure model with 'champion' alias exists first
   ✓ Run the Batch Inference workflow
   ✓ Check the BatchInference notebook logs for errors

5. GitHub Actions Workflow:
   ✓ Verify DEPLOY_AS_MODEL parameter is set correctly
   ✓ Check GitHub Actions logs for job execution status
   ✓ Ensure all required parameters are passed to workflows
"""
)

print("=" * 80)
print("✅ Verification Complete")
print("=" * 80)